In [15]:
from google.colab import drive
import pandas as pd
from statsmodels.tsa.stattools import adfuller,grangercausalitytests
from statsmodels.tsa.ardl import ARDL
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')
df = pd.read_excel('/content/drive/MyDrive/colab đại học/môn kinh tế lượng tài chính/отчёт 2/data kyrgys gốc.xlsx')
df.columns = df.columns.str.replace(' ', '_')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Phần 2
# 2.1 test ADF cho 3 loại chuỗi
print('level:')
for i in ['final_consumption_expenditure','household_consumption','government_consumption','gross_capital_formation',
          'export','import']:
  series = df[i]
  adf = adfuller(series, maxlag = None, regression ='c',autolag= 'AIC')
  print(f'{i} : {adf[1]}')

print('\n difference 1 and 2')
for i in ['final_consumption_expenditure','household_consumption','government_consumption','gross_capital_formation',
          'export','import']:
  for d in [1,2]:
    series = df[i].diff(d).dropna()
    adf = adfuller(series, maxlag = None, regression ='c',autolag= 'AIC')
    print(f'{i}, Bậc{d} : {adf[1]}')

# 2.2 test granger
# chỉ test các biến dừng ở bậc 1 (bỏ qua x6 dù sao cũng ko dùng)
print('\n Test granger')
for i in ['household_consumption','gross_capital_formation','export']:
  granger= grangercausalitytests(df[['GDP',i]].diff(1).dropna(), maxlag=1,verbose = False)
  print(i,granger[1][0]['ssr_ftest'][1])

# 2.3 model DL
from statsmodels.tsa.ardl import ARDL
df_diff = df[['GDP','household_consumption','export']].diff(1).dropna()
df_diff.columns = ['d_gdp', 'd_x2', 'd_x5']
model = ARDL(endog=df_diff['d_gdp'], lags=0, exog=df_diff[['d_x2', 'd_x5']], order={'d_x5': 0, 'd_x2':1}).fit()
print(model.summary())

level:
final_consumption_expenditure : 0.999083769757972
household_consumption : 0.9988562840335837
government_consumption : 0.10450907350456295
gross_capital_formation : 1.0
export : 0.4230687248571628
import : 0.9890612517580399

 difference 1 and 2
final_consumption_expenditure, Bậc1 : 0.14894126886050474
final_consumption_expenditure, Bậc2 : 0.027950554379016338
household_consumption, Bậc1 : 0.010753689828542664
household_consumption, Bậc2 : 0.010543481127306725
government_consumption, Bậc1 : 0.23783746309099035
government_consumption, Bậc2 : 0.0010130197713919186
gross_capital_formation, Bậc1 : 0.002375613889928353
gross_capital_formation, Bậc2 : 2.0556330558250153e-05
export, Bậc1 : 1.024076959630388e-09
export, Bậc2 : 7.835715004051824e-05
import, Bậc1 : 0.6089711429864894
import, Bậc2 : 0.8575777583202917

 Test granger
household_consumption 0.027098550074913883
gross_capital_formation 0.12169184001491241
export 0.0006234387300263639
                              ARDL Model Res

In [33]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 13.0 MB/s eta 0:00:00


In [40]:
from linearmodels import PooledOLS, PanelOLS
import statsmodels.api as sm

panel = pd.read_excel('/content/drive/MyDrive/colab đại học/môn kinh tế lượng tài chính/отчёт 2/Panel data отчёт 2.xlsx')
panel.columns = panel.columns.str.replace(' ', '_').str.lower()
panel = panel.set_index(['country_name','year'])

# pooled
y = panel['gdp']
x = panel.drop('gdp', axis =1)
x = sm.add_constant(x)
ols = PooledOLS(y, x).fit()
print(ols.summary)

# FE
fe = PanelOLS(dependent = y, exog = x, entity_effects= True, time_effects= True).fit() # nếu ko bật các effect nó sẽ chạy y hệt pooled
print(fe.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    gdp   R-squared:                        0.9939
Estimator:                  PooledOLS   R-squared (Between):              0.9992
No. Observations:                 160   R-squared (Within):               0.9809
Date:                Sat, Jun 06 2026   R-squared (Overall):              0.9939
Time:                        09:03:01   Log-likelihood                   -1575.2
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      4166.2
Entities:                           5   P-value                           0.0000
Avg Obs:                       32.000   Distribution:                   F(6,153)
Min Obs:                       32.000                                           
Max Obs:                       32.000   F-statistic (robust):             4166.2
                            